## Cell 1 — Imports and config

In [1]:
"""Phase 2.3 — LLM baselines using Claude.

We score the same test set with two Claude models (Haiku and Sonnet) and
add them to the comparison table from Phase 2.2. Each prediction is
captured as a structured-output tool call so the label is type-safe.

Strategy:
1. Smoke test on 10 test examples with Haiku — verify the prompt and the
   structured-output schema work end-to-end before committing budget.
2. Full Haiku run on all 578 test examples.
3. Full Sonnet run on all 578 test examples.

Cost projection: ~$0.50 Haiku + ~$1.50 Sonnet = ~$2.00 total.
"""
import asyncio
import json
import os
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
from anthropic import AsyncAnthropic
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tenacity import (
    AsyncRetrying,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

os.environ["WANDB__SERVICE_WAIT"] = "300"

DATA_DIR = Path("..") / "data" / "issues" / "splits"
ARTIFACT_DIR = Path("..") / "data" / "llm_baseline_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["bug", "feature", "docs", "question"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

# Model IDs — exact strings used by the Anthropic API.
HAIKU_MODEL = "claude-haiku-4-5-20251001"
SONNET_MODEL = "claude-sonnet-4-5"  # we'll verify the exact id when we start

# Concurrency. Anthropic's rate limit handles 5-10 parallel calls comfortably.
MAX_CONCURRENT = 5

# Token budget per call.
MAX_OUTPUT_TOKENS = 200

SEED = 42
np.random.seed(SEED)

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not set — run `set -a; source ../.env; set +a` before launching the notebook"
)
print("environment ready.")

environment ready.


## Cell 2 — Load splits and define the structured-output tool

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open() as f:
        return [json.loads(line) for line in f]


def issue_to_text(row: dict) -> str:
    title = (row.get("title") or "").strip()
    body = (row.get("body") or "").strip()
    # Truncate body to roughly match the DistilBERT 256-token limit.
    # ~6 chars per token for English, so ~1500 chars.
    if len(body) > 1500:
        body = body[:1500] + "\n[... truncated ...]"
    return f"Title: {title}\n\nBody: {body}" if body else f"Title: {title}"


test_rows = load_jsonl(DATA_DIR / "test.jsonl")
test_distribution = Counter(r["label"] for r in test_rows)
print(f"test: {len(test_rows)} examples")
print(f"distribution: {dict(test_distribution)}")


# Structured-output tool. Claude must call this tool, and its arguments
# are the prediction.
CLASSIFY_TOOL = {
    "name": "classify_issue",
    "description": ("Record your classification of the GitHub issue. Choose exactly one label."),
    "input_schema": {
        "type": "object",
        "properties": {
            "label": {
                "type": "string",
                "enum": LABELS,
                "description": (
                    "The classification. Choose:\n"
                    "  bug      — something is broken or behaves incorrectly\n"
                    "  feature  — a new capability or enhancement is requested\n"
                    "  docs     — documentation is unclear, missing, or wrong\n"
                    "  question — the author is asking how something works or "
                    "needs more info to proceed; often this is unclear what "
                    "category it belongs to."
                ),
            },
            "reasoning": {
                "type": "string",
                "description": "One sentence explaining the choice.",
            },
        },
        "required": ["label", "reasoning"],
    },
}


SYSTEM_PROMPT = """You are classifying GitHub issues for the scikit-learn maintainers.

Each issue is one of:
- bug: existing functionality fails, raises an unexpected error, or produces wrong output
- feature: a request for new capability or enhancement
- docs: documentation is missing, unclear, or incorrect; or improvements to docstrings/examples
- question: the author is asking how something works, needs help using the library, or the issue lacks enough detail to categorize (often these are user questions miscategorized as issues)

Edge cases:
- "Improve performance of X" → feature (it's an enhancement)
- "X documentation says Y but does Z" → bug (incorrect docs that mislead users)
- "How do I make X do Y?" → question
- Issues with detailed reproductions, tracebacks, and stack traces → bug

Be concise. Call the classify_issue tool exactly once."""


print(f"\nSYSTEM_PROMPT length: {len(SYSTEM_PROMPT)} chars")
print(f"tool defined: {CLASSIFY_TOOL['name']}")

test: 578 examples
distribution: {'question': 66, 'docs': 149, 'feature': 89, 'bug': 274}

SYSTEM_PROMPT length: 840 chars
tool defined: classify_issue


## Cell 3 — A single-prediction helper with retries

In [3]:
class PredictionError(Exception):
    """Wraps unexpected response shapes from Claude."""


async def classify_one(
    client: AsyncAnthropic,
    model: str,
    issue: dict,
) -> dict:
    """Classify one issue. Returns dict with label, reasoning, usage, raw."""
    text = issue_to_text(issue)

    async for attempt in AsyncRetrying(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=30),
        retry=retry_if_exception_type((Exception,)),
        reraise=True,
    ):
        with attempt:
            response = await client.messages.create(
                model=model,
                max_tokens=MAX_OUTPUT_TOKENS,
                system=SYSTEM_PROMPT,
                tools=[CLASSIFY_TOOL],
                tool_choice={"type": "tool", "name": "classify_issue"},
                messages=[{"role": "user", "content": text}],
            )

            # Walk the content blocks for the tool_use block.
            tool_block = next(
                (b for b in response.content if b.type == "tool_use"),
                None,
            )
            if tool_block is None:
                raise PredictionError(f"No tool_use block in response: {response.content}")

            label = tool_block.input.get("label")
            reasoning = tool_block.input.get("reasoning", "")
            if label not in LABELS:
                raise PredictionError(f"Invalid label '{label}'")

            return {
                "issue_id": issue["issue_id"],
                "true_label": issue["label"],
                "predicted_label": label,
                "reasoning": reasoning,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "model": model,
            }

    raise PredictionError("retries exhausted")  # pragma: no cover


# Smoke test — one example, Haiku.
async def _smoke_one() -> None:
    async with AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"]) as client:
        sample = test_rows[0]
        print(f"sample: {sample['title'][:80]}")
        print(f"true: {sample['label']}")
        result = await classify_one(client, HAIKU_MODEL, sample)
        print(f"predicted: {result['predicted_label']}")
        print(f"reasoning: {result['reasoning']}")
        print(f"tokens: input={result['input_tokens']} output={result['output_tokens']}")


await _smoke_one()

sample: Convergence Warning message!!!
true: question
predicted: question
reasoning: The author is asking for help troubleshooting a convergence warning without providing specific code, error details, or a minimal reproduction case, indicating they need guidance on how to resolve the issue rather than reporting a confirmed bug or feature request.
tokens: input=1023 output=96


## Cell 4  — Concurrent batch prediction with caching

In [4]:
async def classify_batch(
    model: str,
    issues: list[dict],
    output_path: Path,
    description: str,
) -> list[dict]:
    """Classify a batch of issues with concurrency and caching to JSONL."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Load existing predictions to skip them.
    cached: dict[int, dict] = {}
    if output_path.exists():
        with output_path.open() as f:
            for line in f:
                if line.strip():
                    pred = json.loads(line)
                    cached[pred["issue_id"]] = pred
        print(f"  resumed: {len(cached)} predictions already cached")

    todo = [i for i in issues if i["issue_id"] not in cached]
    print(f"  {description}: {len(todo)} new predictions to make")

    if not todo:
        return list(cached.values())

    sem = asyncio.Semaphore(MAX_CONCURRENT)

    async def _one(client: AsyncAnthropic, issue: dict) -> dict | None:
        async with sem:
            try:
                return await classify_one(client, model, issue)
            except Exception as exc:
                print(f"  FAILED issue_id={issue['issue_id']}: {exc}")
                return None

    async with AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"]) as client:
        with output_path.open("a") as f:
            tasks = [_one(client, issue) for issue in todo]
            for i, coro in enumerate(asyncio.as_completed(tasks), start=1):
                result = await coro
                if result is not None:
                    f.write(json.dumps(result) + "\n")
                    f.flush()
                    cached[result["issue_id"]] = result
                if i % 50 == 0:
                    print(f"  {description}: {i}/{len(todo)} ...")

    return list(cached.values())

## Cell 5 — Haiku smoke run on 10 examples

In [5]:
SMOKE_N = 10
smoke_path = ARTIFACT_DIR / "smoke_haiku.jsonl"

if smoke_path.exists():
    smoke_path.unlink()  # always start fresh for the smoke test

print(f"=== smoke test: Haiku on {SMOKE_N} examples ===")
smoke_preds = await classify_batch(
    HAIKU_MODEL,
    test_rows[:SMOKE_N],
    smoke_path,
    description="smoke",
)

# Inspect the results.
correct = sum(1 for p in smoke_preds if p["predicted_label"] == p["true_label"])
print(f"\nsmoke accuracy: {correct}/{len(smoke_preds)}")
for p in smoke_preds[:5]:
    marker = "✓" if p["predicted_label"] == p["true_label"] else "✗"
    print(
        f"  {marker} true={p['true_label']:<10} pred={p['predicted_label']:<10}  reasoning: {p['reasoning'][:60]}"
    )

total_in = sum(p["input_tokens"] for p in smoke_preds)
total_out = sum(p["output_tokens"] for p in smoke_preds)
print(f"\nsmoke total tokens: input={total_in} output={total_out}")
cost = total_in * 1.0 / 1e6 + total_out * 5.0 / 1e6
print(f"smoke cost: ${cost:.4f}")

=== smoke test: Haiku on 10 examples ===
  smoke: 10 new predictions to make

smoke accuracy: 7/10
  ✓ true=question   pred=question    reasoning: The author is asking for help with a convergence warning the
  ✓ true=docs       pred=docs        reasoning: This is a Sphinx documentation build issue where cross-refer
  ✓ true=docs       pred=docs        reasoning: The issue explicitly requests improvements to the documentat
  ✓ true=question   pred=question    reasoning: This is a Request for Comments (RFC) asking the maintainers 
  ✓ true=bug        pred=bug         reasoning: This is a performance regression (not an enhancement) in exi

smoke total tokens: input=13310 output=907
smoke cost: $0.0178


## Cell 6 — Haiku full run on all 578 test examples

In [ ]:
haiku_path = ARTIFACT_DIR / "predictions_haiku.jsonl"

print("=== Haiku full run on 578 test examples ===")
print("  estimated cost: ~$1.00")
print()

haiku_preds = await classify_batch(
    HAIKU_MODEL,
    test_rows,
    haiku_path,
    description="haiku",
)
print(f"\nhaiku predictions complete: {len(haiku_preds)} / {len(test_rows)}")


# Compute metrics.
def compute_metrics(preds: list[dict], model_name: str) -> dict:
    """Compute accuracy, macro-F1, per-class F1, confusion matrix."""
    # Sort by issue_id for stable ordering with the test set.
    by_id = {p["issue_id"]: p for p in preds}
    aligned = [by_id[r["issue_id"]] for r in test_rows if r["issue_id"] in by_id]
    if len(aligned) != len(test_rows):
        print(f"  WARNING: only {len(aligned)}/{len(test_rows)} aligned predictions")

    y_true = np.array([LABEL_TO_ID[p["true_label"]] for p in aligned])
    y_pred = np.array([LABEL_TO_ID[p["predicted_label"]] for p in aligned])

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    per_class = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        output_dict=True,
        zero_division=0,
    )
    per_class_f1 = {label: per_class[label]["f1-score"] for label in LABELS}
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))

    total_in = sum(p["input_tokens"] for p in aligned)
    total_out = sum(p["output_tokens"] for p in aligned)

    return {
        "model": model_name,
        "n_examples": len(aligned),
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "per_class_f1": per_class_f1,
        "confusion_matrix": cm.tolist(),
        "total_input_tokens": total_in,
        "total_output_tokens": total_out,
    }


haiku_metrics = compute_metrics(haiku_preds, model_name=HAIKU_MODEL)
print(f"\nHaiku — accuracy: {haiku_metrics['accuracy']:.4f}")
print(f"Haiku — macro-F1: {haiku_metrics['macro_f1']:.4f}")
print("Haiku — per-class F1:")
for label, f1 in haiku_metrics["per_class_f1"].items():
    print(f"  {label:10s}: {f1:.4f}")

# Cost.
haiku_cost = (
    haiku_metrics["total_input_tokens"] * 1.0 / 1e6
    + haiku_metrics["total_output_tokens"] * 5.0 / 1e6
)
print(f"\nHaiku — total cost: ${haiku_cost:.4f}")
print(f"Haiku — per 1k issues: ${haiku_cost * 1000 / 578:.4f}")

=== Haiku full run on 578 test examples ===
  estimated cost: ~$1.00

  haiku: 578 new predictions to make
  haiku: 50/578 ...
  haiku: 100/578 ...
  haiku: 150/578 ...
  haiku: 200/578 ...
  haiku: 250/578 ...
  haiku: 300/578 ...
  haiku: 350/578 ...
  haiku: 400/578 ...
  haiku: 450/578 ...
  haiku: 500/578 ...
  haiku: 550/578 ...

haiku predictions complete: 578 / 578

Haiku — accuracy: 0.8495
Haiku — macro-F1: 0.7664
Haiku — per-class F1:
  bug       : 0.9122
  feature   : 0.7958
  docs      : 0.8881
  question  : 0.4694

Haiku — total cost: $1.0630
Haiku — per 1k issues: $1.8391


## Cell 7 — Sonnet full run

In [ ]:
sonnet_path = ARTIFACT_DIR / "predictions_sonnet.jsonl"

print("=== Sonnet full run on 578 test examples ===")
print("  estimated cost: ~$3.00")
print()

sonnet_preds = await classify_batch(
    SONNET_MODEL,
    test_rows,
    sonnet_path,
    description="sonnet",
)
print(f"\nsonnet predictions complete: {len(sonnet_preds)} / {len(test_rows)}")

sonnet_metrics = compute_metrics(sonnet_preds, model_name=SONNET_MODEL)
print(f"\nSonnet — accuracy: {sonnet_metrics['accuracy']:.4f}")
print(f"Sonnet — macro-F1: {sonnet_metrics['macro_f1']:.4f}")
print("Sonnet — per-class F1:")
for label, f1 in sonnet_metrics["per_class_f1"].items():
    print(f"  {label:10s}: {f1:.4f}")

sonnet_cost = (
    sonnet_metrics["total_input_tokens"] * 3.0 / 1e6
    + sonnet_metrics["total_output_tokens"] * 15.0 / 1e6
)
print(f"\nSonnet — total cost: ${sonnet_cost:.4f}")
print(f"Sonnet — per 1k issues: ${sonnet_cost * 1000 / 578:.4f}")

=== Sonnet full run on 578 test examples ===
  estimated cost: ~$3.00

  sonnet: 578 new predictions to make
  sonnet: 50/578 ...
  sonnet: 100/578 ...
  sonnet: 150/578 ...
  sonnet: 200/578 ...
  sonnet: 250/578 ...
  sonnet: 300/578 ...
  sonnet: 350/578 ...
  sonnet: 400/578 ...
  sonnet: 450/578 ...
  sonnet: 500/578 ...
  sonnet: 550/578 ...

sonnet predictions complete: 578 / 578

Sonnet — accuracy: 0.8114
Sonnet — macro-F1: 0.7329
Sonnet — per-class F1:
  bug       : 0.8924
  feature   : 0.7729
  docs      : 0.8199
  question  : 0.4464

Sonnet — total cost: $3.1236
Sonnet — per 1k issues: $5.4042


## Cell 8 — Build the comparison report and write artifacts

In [8]:
# Load Phase 2.1 (DistilBERT) and Phase 2.2 (classical) model cards
distilbert_card = json.loads(
    (Path("..") / "data" / "classifier_artifacts" / "model_card.json").read_text()
)
classical_card = json.loads(
    (Path("..") / "data" / "classical_baseline_artifacts" / "comparison_report.json").read_text()
)

# Build the four-way comparison table.
rows = [
    {
        "name": "Classical (TF-IDF + LogReg)",
        "test_accuracy": classical_card["test_accuracy"],
        "test_macro_f1": classical_card["test_macro_f1"],
        "per_class_f1": classical_card["per_class_f1"],
        "cost_per_1k_issues_usd": 0.0,
        "notes": "Trained locally; no inference cost.",
    },
    {
        "name": "DistilBERT (fine-tuned)",
        "test_accuracy": distilbert_card["test_accuracy"],
        "test_macro_f1": distilbert_card["test_macro_f1"],
        "per_class_f1": distilbert_card["per_class_f1"],
        "cost_per_1k_issues_usd": 0.0,
        "notes": "Inference cost = container CPU time only.",
    },
    {
        "name": "Haiku 4.5",
        "test_accuracy": haiku_metrics["accuracy"],
        "test_macro_f1": haiku_metrics["macro_f1"],
        "per_class_f1": haiku_metrics["per_class_f1"],
        "cost_per_1k_issues_usd": haiku_cost * 1000 / 578,
        "notes": (
            f"input ${haiku_metrics['total_input_tokens'] / 1e6:.3f}M tokens @ $1.00, "
            f"output ${haiku_metrics['total_output_tokens'] / 1e6:.3f}M tokens @ $5.00"
        ),
    },
    {
        "name": "Sonnet 4.6",
        "test_accuracy": sonnet_metrics["accuracy"],
        "test_macro_f1": sonnet_metrics["macro_f1"],
        "per_class_f1": sonnet_metrics["per_class_f1"],
        "cost_per_1k_issues_usd": sonnet_cost * 1000 / 578,
        "notes": (
            f"input ${sonnet_metrics['total_input_tokens'] / 1e6:.3f}M tokens @ $3.00, "
            f"output ${sonnet_metrics['total_output_tokens'] / 1e6:.3f}M tokens @ $15.00"
        ),
    },
]

# Print as a table.
print(f"\n{'Classifier':<32} {'Acc':>8} {'F1':>8} {'F1-q':>8} {'$/1k':>8}")
print("-" * 70)
for r in rows:
    print(
        f"{r['name']:<32} "
        f"{r['test_accuracy']:>8.4f} "
        f"{r['test_macro_f1']:>8.4f} "
        f"{r['per_class_f1']['question']:>8.4f} "
        f"{r['cost_per_1k_issues_usd']:>8.2f}"
    )

# Save the report.
report = {
    "comparison": rows,
    "winner_on_macro_f1": max(rows, key=lambda r: r["test_macro_f1"])["name"],
    "winner_on_question_class": max(rows, key=lambda r: r["per_class_f1"]["question"])["name"],
    "deployment_recommendation": "Haiku 4.5 — best macro-F1 and cheapest of the LLMs",
    "generated_at": datetime.now(UTC).isoformat(),
    "test_set": {
        "n": len(test_rows),
        "training_data_sha256": distilbert_card["training_data_sha256"],
    },
    "haiku_full_metrics": haiku_metrics,
    "sonnet_full_metrics": sonnet_metrics,
}

(ARTIFACT_DIR / "comparison_report.json").write_text(json.dumps(report, indent=2))
print(f"\nsaved {ARTIFACT_DIR / 'comparison_report.json'}")
print(f"\nWINNER on macro-F1: {report['winner_on_macro_f1']}")
print(f"WINNER on question class: {report['winner_on_question_class']}")


Classifier                            Acc       F1     F1-q     $/1k
----------------------------------------------------------------------
Classical (TF-IDF + LogReg)        0.8201   0.6977   0.2558     0.00
DistilBERT (fine-tuned)            0.8478   0.7462   0.3600     0.00
Haiku 4.5                          0.8495   0.7664   0.4694     1.84
Sonnet 4.6                         0.8114   0.7329   0.4464     5.40

saved ../data/llm_baseline_artifacts/comparison_report.json

WINNER on macro-F1: Haiku 4.5
WINNER on question class: Haiku 4.5
